# Analysdemo – säker serverbaserad analys

All data stannar i klustret. Inloggningsuppgifter till datalagret injiceras som miljövariabler (`PG*`) från en personlig Secret – inga lösenord i kod.

In [ ]:
import os
print({k: ('***' if 'PASSWORD' in k else v) for k, v in os.environ.items() if k.startswith('PG')})

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Tom URL -> libpq läser PGHOST/PGUSER/PGPASSWORD/... från miljön
engine = create_engine('postgresql+psycopg://')
engine

## 1. Rådata
Allt som visas här skickas till webbläsaren. **Det är exfiltrationsrisken** – skärmdump, kopiera, ladda ner.

In [ ]:
df = pd.read_sql('SELECT * FROM personer', engine)
print(len(df), 'rader')
df.head(20)

## 2. Aggregerat resultat med röjandekontroll
Celler med färre än 5 individer undertrycks (statistical disclosure control). Det här är vad som borde få lämna miljön.

In [ ]:
agg = (df.groupby(['kommun', 'diagnoskod'])
         .agg(antal=('personnr', 'count'), medelinkomst=('inkomst', 'mean'))
         .reset_index())
agg.loc[agg.antal < 5, ['antal', 'medelinkomst']] = None
agg.round(0)

## 3. Egress-kontroll
EgressFirewall tillåter bara pypi/open-vsx/github. Allt annat ut ur klustret blockeras.

In [ ]:
import urllib.request
for url in ['https://pypi.org/simple/', 'https://example.com', 'https://pastebin.com']:
    try:
        urllib.request.urlopen(url, timeout=5)
        print('OK     ', url)
    except Exception as e:
        print('BLOCKAD', url, '-', type(e).__name__)

## 4. Export
En CSV skapas i workspacen. Visa att den *går* att ladda ner via editorn (högerklick → Download) – det är gapet som en airlock ska stänga.

In [ ]:
os.makedirs('export', exist_ok=True)
df.to_csv('export/radata.csv', index=False)
agg.to_csv('export/aggregat.csv', index=False)
!ls -lh export/